In [1]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torch
import torchvision
from sklearn.metrics import accuracy_score

In [2]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [3]:
class ImageDataset(Dataset):
    def __init__(self, image_dir, csv_path, transform=None, label_col=True):
        """
        Args:
            image_dir (str): Directory with training images.
            csv_path (str): Path to CSV with image filenames and labels.
            transform (callable, optional): Optional transform to apply to each image.
        """
        self.image_dir = image_dir
        self.labels_df = pd.read_csv(csv_path)
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        # Get filename and label
        img_name = self.labels_df.iloc[idx, 0]
        if self.label_col:
            label = int(self.labels_df.iloc[idx, 1])
        else:
            label = 0

        # Build full path and open image
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        # Apply any transforms
        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
image_dir = "image-classification-real-or-ai-generated-photo/train/train"
csv_path = "image-classification-real-or-ai-generated-photo/train.csv"

In [5]:
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     # transforms.Resize((384, 384)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                             std=[0.229, 0.224, 0.225])
# ])

In [6]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(380, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.RandomAdjustSharpness(sharpness_factor=1.5, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transforms = transforms.Compose([
    transforms.Resize(400),          # Resize shorter side to 400 px (maintains aspect ratio)
    transforms.CenterCrop(380),      # Crop the center 380×380 region
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [7]:
# dataset = ImageDataset(image_dir=image_dir, csv_path=csv_path, transform=transform)
dataset = ImageDataset(image_dir=image_dir, csv_path=csv_path, transform=train_transforms)

In [8]:
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2])

In [9]:
mini_batch_size = 32

In [10]:
train_dl = DataLoader(train_dataset, batch_size=mini_batch_size, shuffle=True, drop_last=False)
val_dl = DataLoader(val_dataset, batch_size=mini_batch_size * 2, drop_last=False)

In [11]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [12]:
def fit(model, optimizer, early_stopping, train_dl, valid_dl):
    loss_func = torch.nn.CrossEntropyLoss()

    # loop over epochs
    for epoch in range(100):
        model.train()

        # loop over mini-batches
        for X_mb, y_mb in train_dl:
            X_mb = X_mb.to(device)
            y_mb = y_mb.to(device)
            y_hat = model(X_mb)

            loss = loss_func(y_hat, y_mb)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        model.eval()
        with torch.no_grad():
            train_loss = sum(loss_func(model(X_mb.to(device)), y_mb.to(device)) for X_mb, y_mb in train_dl)
            valid_loss = sum(loss_func(model(X_mb.to(device)), y_mb.to(device)) for X_mb, y_mb in valid_dl)
        print('epoch {}, training loss {}'.format(epoch + 1, train_loss / len(train_dl)))
        print('epoch {}, validation loss {}'.format(epoch + 1, valid_loss / len(valid_dl)))

        early_stopping(valid_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

In [13]:
# model = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
# num_features = model.classifier[2].in_features
# model.classifier[2] = torch.nn.Linear(num_features, 2)

In [14]:
model = torchvision.models.resnet50(weights='IMAGENET1K_V1')
model.fc = torch.nn.Linear(model.fc.in_features, 2)

In [15]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [16]:
model = model.to(device)

In [17]:
# optimizer = torch.optim.Adam(model.parameters())
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

early_stopping = EarlyStopping(patience=10, delta=0.01)

In [18]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

23512130

In [19]:
fit(model, optimizer, early_stopping, train_dl, val_dl)

epoch 1, training loss 0.1560535877943039
epoch 1, validation loss 0.2669928967952728
epoch 2, training loss 0.16666655242443085
epoch 2, validation loss 0.25400951504707336
epoch 3, training loss 0.04770031198859215
epoch 3, validation loss 0.16928112506866455
epoch 4, training loss 0.03348332270979881
epoch 4, validation loss 0.29635605216026306
epoch 5, training loss 0.033680450171232224
epoch 5, validation loss 0.19130225479602814
epoch 6, training loss 0.015455145388841629
epoch 6, validation loss 0.14116214215755463
epoch 7, training loss 0.01658066362142563
epoch 7, validation loss 0.09261558204889297
epoch 8, training loss 0.02549796737730503
epoch 8, validation loss 0.20699620246887207
epoch 9, training loss 0.04461778700351715
epoch 9, validation loss 0.17168909311294556
epoch 10, training loss 0.04571491852402687
epoch 10, validation loss 0.14996127784252167
epoch 11, training loss 0.03602515533566475
epoch 11, validation loss 0.297197550535202
epoch 12, training loss 0.0151

In [20]:
early_stopping.load_best_model(model)

In [21]:
model.eval()
predictions = []
true_labels = []

with torch.no_grad():  # disable gradient tracking
    for X_batch, y_batch in train_dl:
        outputs = model(X_batch.to(device)).cpu()
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.tolist())
        true_labels.extend(y_batch.tolist())

In [22]:
accuracy_score(predictions, true_labels)

0.996031746031746

In [23]:
model.eval()
predictions = []
true_labels = []

with torch.no_grad():  # disable gradient tracking
    for X_batch, y_batch in val_dl:
        outputs = model(X_batch.to(device)).cpu()
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.tolist())
        true_labels.extend(y_batch.tolist())

In [24]:
accuracy_score(predictions, true_labels)

0.9523809523809523

In [25]:
# test_dataset = ImageDataset(image_dir="image-classification-real-or-ai-generated-photo/test/test", csv_path="image-classification-real-or-ai-generated-photo/test.csv", transform=transform, label_col=False)
test_dataset = ImageDataset(image_dir="image-classification-real-or-ai-generated-photo/test/test", csv_path="image-classification-real-or-ai-generated-photo/test.csv", transform=test_transforms, label_col=False)

test_dl = DataLoader(test_dataset, batch_size=mini_batch_size * 2, drop_last=False)

In [26]:
model.eval()
predictions = []

with torch.no_grad():  # disable gradient tracking
    for X_batch, y_batch in test_dl:
        outputs = model(X_batch.to(device)).cpu()
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.tolist())

In [27]:
test_csv = pd.read_csv("image-classification-real-or-ai-generated-photo/test.csv")
test_csv["Label"] = predictions

In [28]:
test_csv

,Image,Label
0,946.jpg,0
1,947.jpg,1
2,948.jpg,1
3,949.jpg,1
4,950.jpg,0
...,...,...
397,1343.jpg,1
398,1344.jpg,1
399,1345.jpg,1
400,1346.jpg,1


In [29]:
test_csv.Label.mean()

np.float64(0.5248756218905473)

In [30]:
test_csv.to_csv("submission.csv", index=False)